# Справочник датасетов дашборда

Все витрины дашборда лежат в `data/mart/` в формате CSV. Ниже — перечень
датасетов, число строк и полей, а также описание каждого поля (из
`data/reference/field_dictionary.json`).

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

_root = Path.cwd()
while not (_root / "data" / "raw").exists() and _root.parent != _root:
    _root = _root.parent
sys.path.insert(0, str(_root))
PROJECT_ROOT = _root
MART = PROJECT_ROOT / "data" / "mart"
fd = json.loads((PROJECT_ROOT / "data" / "reference" / "field_dictionary.json").read_text(encoding="utf-8"))
field_meta = {f["field"]: f for f in fd["fields"]}

DATASETS = [
    ("resort_scores.csv", "Рейтинг: 13 баллов, 5 компонентов, индекс, место, качество"),
    ("resort_features.csv", "Исходные признаки (подмножество resort_scores)"),
    ("resort_components.csv", "Компоненты рейтинга (курорт × компонент)"),
    ("ranking_stability.csv", "Устойчивость, локальный рейтинг, сценарии весов"),
    ("resort_reasons.csv", "Объяснения «почему» (длинный формат)"),
    ("catalog_scenarios.csv", "Сценарии каталога и их KPI"),
    ("catalog_members.csv", "Состав каталога по сценариям"),
    ("climate_monthly.csv", "Месячный климат ноябрь–апрель"),
    ("weather_forecast_daily.csv", "Прогноз 7 дней (курорт × дата)"),
]


def summarize(name):
    df = pd.read_csv(MART / name)
    return len(df), list(df.columns)


## Сводка

In [ ]:
rows = []
for name, desc in DATASETS:
    n, cols = summarize(name)
    rows.append({"Файл": name, "Назначение": desc, "Строк": n, "Полей": len(cols)})
print(pd.DataFrame(rows).to_string(index=False))

## Поля по датасетам

In [ ]:
for name, desc in DATASETS:
    n, cols = summarize(name)
    print(f"\n### {name} — {desc}")
    print(f"{n} строк, {len(cols)} полей")
    for c in cols:
        m = field_meta.get(c)
        if m:
            unit = m.get("unit") or "—"
            caveat = f" | {m['caveat']}" if m.get("caveat") else ""
            print(f"  • {c} — {m['label_ru']} [{unit}]{caveat}")
        else:
            print(f"  • {c} — справочное поле/ключ")

## Группы полей

In [ ]:
for g in fd["groups"]:
    print(f"- {g['group']} — {g['description']}")